Import big matrix

In [83]:
import pandas as pd

bm_og = pd.read_csv(
    '/home/sulay/deep-linear-bandits/kuairec/data/big_matrix.csv',
        usecols=[
            'user_id',
            'video_id',
            'watch_ratio'
        ]
    ).sort_values(by=['user_id', 'video_id'])

bm_og

,user_id,video_id,watch_ratio
312,0,42,1.098951
292,0,67,2.759635
275,0,80,1.188017
61,0,110,1.408627
318,0,128,1.281867
...,...,...,...
12530627,7175,10552,0.147431
12530710,7175,10572,0.293611
12530711,7175,10572,0.293611
12530717,7175,10589,0.560359


In [75]:
bm_og.duplicated(subset=['user_id', 'video_id']).sum()

np.int64(2229837)

In [84]:
bm_f = bm_og.copy()

bm_f = bm_f.groupby(by=['user_id', 'video_id'], as_index=False).max()

bm_f

,user_id,video_id,watch_ratio
0,0,42,1.098951
1,0,67,2.759635
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


In [85]:
bm_f.groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,80,80
4604,125,125
3935,151,151
3175,155,155
268,156,156
...,...,...
6565,3341,3341
816,3348,3348
5412,3353,3353


In [86]:
bm_f[bm_f.watch_ratio < 2.0].groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,63,63
4604,102,102
3935,121,121
3175,133,133
4812,138,138
...,...,...
6565,3295,3295
2735,3303,3303
5412,3308,3308


In [87]:
bm_f.groupby(by='video_id').count().sort_values(by='watch_ratio')

,user_id,watch_ratio
video_id,,
11,1,1
4943,1,1
40,1,1
43,1,1
10705,1,1
...,...,...
1507,5168,5168
8145,5175,5175
1037,5211,5211


Filter for the strong positive interactions to train on

In [88]:
bm = bm_f.copy()

bm = bm[bm.watch_ratio >= 2.0]

bm

,user_id,video_id,watch_ratio
1,0,67,2.759635
6,0,133,2.458447
10,0,152,2.326087
11,0,154,4.353647
15,0,171,33.276021
...,...,...,...
10300855,7175,9811,2.232981
10300862,7175,9841,2.167814
10300910,7175,10130,15.828342
10300955,7175,10408,9.241597


In [277]:
liked_counts = bm.groupby(by='user_id')['user_id'].count()

liked_counts

user_id
0       258
1       166
2        38
3       163
4        51
       ... 
7171    134
7172    271
7173     95
7174     37
7175    223
Name: user_id, Length: 7175, dtype: int64

In [106]:
liked_counts.describe()

count    7175.000000
mean      117.967108
std       101.901022
min         1.000000
25%        43.000000
50%        88.000000
75%       162.000000
max      1002.000000
Name: user_id, dtype: float64

In [89]:
bm_f[bm_f.watch_ratio < 2.0]

,user_id,video_id,watch_ratio
0,0,42,1.098951
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
5,0,130,0.079565
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


Split into training & validation

In [270]:
from sklearn.model_selection import train_test_split

low = bm.groupby('user_id')['user_id'].transform('count') < 5

nonlow = bm[~low]
bm_train, bm_val = train_test_split(
    nonlow,
    train_size=0.8,
    shuffle=True,
    stratify=nonlow['user_id']
)
bm_train = pd.concat([bm_train, bm[low]])

bm_train, bm_val

(         user_id  video_id  watch_ratio
 3076689     2142      6789     6.415015
 2814127     1953      6110    13.719355
 1537090     1053      4327     2.004722
 9929966     6916      1327     5.677419
 4969110     3450      3160     5.360320
 ...          ...       ...          ...
 9176402     6384      3632     3.930262
 9555229     6658      3400     2.603915
 9555261     6658      4374     2.519602
 9555297     6658      5047     2.600979
 9555519     6658     10653     2.045629
 
 [677144 rows x 3 columns],
          user_id  video_id  watch_ratio
 2049241     1415      7085     2.133833
 5180427     3596      7244     2.801001
 6353481     4391      8728     2.961000
 7251050     5021      1903     2.546499
 5826557     4049       600     2.269346
 ...          ...       ...          ...
 8917970     6196     10424     2.082573
 9030346     6286      5575     3.744017
 3896633     2729      4977     2.294519
 709712       473      8418     2.478042
 4865677     3388      1573

Convert to PyTorch dataset format

In [288]:
from torch.utils.data import Dataset

class KRDataset(Dataset):
    def __init__(self, data):
        self.user_ids = data['user_id'].to_numpy()
        self.item_ids = data['video_id'].to_numpy()
        
        # Compute user positive sets for rejection sampling negatives
        self.positives = [set() for _ in range(7176)]
        for i in range(len(self.user_ids)):
            self.positives[self.user_ids[i]].add(self.item_ids[i])

    def __len__(self):
        return len(self.user_ids)
    
    def __getitem__(self, idx):
        return {
            'user_id': self.user_ids[idx],
            'pos_item_id': self.item_ids[idx]
        }

bm_t = KRDataset(bm_train)
bm_v = KRDataset(bm_val)

print(len(bm_t))
print(len(bm_v))

677144
169270


In [296]:
# Checking positives are correct
t = 0
counts = bm_train.groupby(by='user_id')['user_id'].count()
for i in range(7176):
    p = bm_t.positives[i]

    if len(p) == 0:
        continue
    
    if len(p) != counts[i]:
        print("Mismatch!")

    t += len(p)
print(f"Total positives: {t}")
print(sorted(bm_t.positives[17]))
print(bm_train[bm_train.user_id == 17]['video_id'].sort_values())

Total positives: 677144
[np.int64(151), np.int64(164), np.int64(179), np.int64(183), np.int64(229), np.int64(238), np.int64(297), np.int64(300), np.int64(336), np.int64(416), np.int64(423), np.int64(429), np.int64(525), np.int64(558), np.int64(569), np.int64(580), np.int64(590), np.int64(601), np.int64(622), np.int64(684), np.int64(686), np.int64(715), np.int64(721), np.int64(745), np.int64(799), np.int64(804), np.int64(835), np.int64(839), np.int64(840), np.int64(859), np.int64(885), np.int64(901), np.int64(918), np.int64(1002), np.int64(1009), np.int64(1019), np.int64(1044), np.int64(1101), np.int64(1116), np.int64(1137), np.int64(1182), np.int64(1254), np.int64(1288), np.int64(1299), np.int64(1306), np.int64(1314), np.int64(1349), np.int64(1370), np.int64(1410), np.int64(1445), np.int64(1511), np.int64(1620), np.int64(1623), np.int64(1703), np.int64(1709), np.int64(1965), np.int64(1973), np.int64(2077), np.int64(2113), np.int64(2117), np.int64(2201), np.int64(2242), np.int64(2273), 

Set up two-tower

In [273]:
import torch
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.u = nn.Embedding(7176, 32)
        self.i = nn.Embedding(10728, 32)

    def forward(self, u_ids, i_ids):
        u = self.u(u_ids)
        i = self.i(i_ids)

        return u @ i.T

model = TwoTower().to(device)

model

TwoTower(
  (u): Embedding(7176, 32)
  (i): Embedding(10728, 32)
)

Train the two-tower and see validation loss each epoch

In [274]:
from torch.utils.data import DataLoader

bm_train_ldr = DataLoader(bm_t, batch_size=512, shuffle=True)
bm_val_ldr = DataLoader(bm_v, batch_size=512, shuffle=True)